# core

> Scheme implementation in Python

`eval`, `apply`, and `env` are the central loop of a Lisp interpreter:

- **`env`** answers: “what does this symbol mean?”
- **`eval`** answers: “what is the value of this expression in this environment?”
- **`apply`** answers: “given a function/procedure and evaluated arguments, how do I call it?”

For a simple expression:

```scheme
(+ x 3)
```

the flow is:

1. `eval` sees a list, so it treats it as a procedure call.
2. It evaluates the first element, `+`, by looking it up in `env`.
3. It evaluates each argument: `x` is looked up in `env`, `3` evaluates to itself.
4. Then `eval` hands the resulting procedure and values to `apply`.
5. `apply` actually calls the procedure.

Conceptually:

```python
proc = eval('+', env)
args = [eval('x', env), eval(3, env)]
return apply(proc, args)
```

So `eval` walks and interprets expressions; `apply` performs calls; `env` gives meaning to symbols along the way.

In [ ]:
#| default_exp core

In [ ]:
#| export
import math, operator as op
from fastcore.basics import store_attr, first

from compact.reader import *
from compact.types import *

### Environment

In Lisp/Scheme, an **env** is the “memory of names”: it maps symbols like `x`, `+`, or `square` to the values they currently mean.

For example, when evaluating:

```scheme
(+ x 3)
```

the evaluator sees `+` and `x` as symbols. It needs an environment to answer:

- what function does `+` refer to?
- what value does `x` refer to?

So conceptually:

```python
env = {'x': 10, '+': operator.add}
```

Then `(+ x 3)` can become “call add on 10 and 3”.

The important idea is: **expressions don’t carry all their meaning alone; symbols get their meaning from the environment they’re evaluated in.**

Later, `env` also lets us support local bindings:

```scheme
(let ((x 5))
  (+ x 1))
```

Inside the `let`, `x` means `5`; outside, it might mean something else. That means environments often form a **chain**: look in the local env first, then the outer env if not found.

_In our case since the goal is to produce an embedded lisp that interops with python, we are going to share globals() (the python symbols) with env used by the lisp evaluator._

In [ ]:
#| export
class Env:
    "implementation of env for scheme eval()."
    def __init__(self, d, primitives:dict=(), parent=None): store_attr(cast=True)

    def __getitem__(self, k):
        s = self.find(k)
        if s is not None: 
            # we currently do not allow shadowing of primitives - this is not ideal
            if k in s.primitives: return s.primitives[k]
            return s.d[k]
        raise KeyError(k)

    def __setitem__(self, k, v): self.d[k] = v

    def __contains__(self, k): return self.find(k) is not None

    def update(self, d): self.primitives.update(d)

    def find(self, k):
        if k in self.d | self.primitives: return self
        if self.parent is not None: return self.parent.find(k)
        return None
    
    def is_primitive(self, k):
        if self.parent is None: return k in self.primitives
        return self.parent.is_primitive(k)

    def new_frame(self, bindings=()):
        return Env(dict(bindings), parent=self)


### The Evaluator: eval and apply
The two-function core: `eval` dispatches, `apply` calls.

In [ ]:
#| export
def scm_apply(fn, args): return fn(*args)

In [ ]:
#| export
def scm_eval_one_step(expr, env, sfs=()):
    "eval-uate lisp expressions given an environment and handlers for special forms"
    # Atoms
    if isinstance(expr, Symbol): return env[expr.s]
    if not isinstance(expr, list): return expr

    # lists
    hd, body = expr[0], expr[1:]

    if isinstance(hd, Symbol) and hd.s in sfs: return sfs[hd.s](body, env, sfs)
    fn = scm_eval_tco(hd, env, sfs)
    if isinstance(fn, Macro): return Thunk(fn.fn(*body), env)

    return scm_apply(fn, [scm_eval_tco(o, env, sfs) for o in body])

In [ ]:
#| export
def scm_eval_tco(expr, env, sfs=()):
    "eval but with tail call optimization"
    while True:
        r = scm_eval_one_step(expr, env, sfs)
        if isinstance(r, Thunk): expr, env = r.expr, r.env
        else: return r

Lisp code is mostly recursive. 

Naive recursive evaluation stack overflows for deep recursion (e.g. `(fact 10000)`). Instead of recursing, we use something called Tail Call Optimization which can be used to convert recursion into iteration. Calls return `Thunk`s that can be executed in the same stack frame. This is also called trampolining.

In our case, the main eval loop is split into two parts that call each other and eventually apply.


Note also, we have a dictionary called sfs which will be used to externalize our special forms from our eval (typically these would be `if` clauses inside eval.

### Primitive Procedures

| Operator | Meaning |
|---|---|
| `+` `-` `*` `/` | Arithmetic (variadic; `-` negates if unary, `/` inverts if unary) |
| `=` `<` `>` `<=` `>=` | Numeric comparison |
| `not` | Boolean negation |
| `number?` `string?` `symbol?` `boolean?` | Type predicates |
| `list` `cons` `car` `cdr` | List construction and access |
| `null?` `pair?` `list?` | List predicates |

In [ ]:
#| export
def _scm_sub(x, *xs): return x - sum(xs) if xs else -x
def _scm_div(x, *xs): return 1/x if not xs else x / math.prod(xs)

def _is_num(x): return not isinstance(x, bool) and isinstance(x, (int, float, complex))
def _is_sym(x): return isinstance(x, Symbol)

_builtin = {
    "+": lambda *xs: sum(xs),
    "*": lambda *xs: math.prod(xs),
    "/": _scm_div,
    "-": _scm_sub,
    ">": op.gt,
    "<": op.lt,
    "=": lambda x,y: _is_num(x) and _is_num(y) and x == y,
    "<=": op.le,
    ">=": op.ge,

    "number?": _is_num,
    "string?": lambda x: isinstance(x, str),
    "symbol?": _is_sym,
    "boolean?": lambda x: isinstance(x, bool),
}

def _is_list(x): return isinstance(x, list)
def _is_pair(x): return isinstance(x, list) and bool(x)

_builtin |= {
    "list": lambda *xs: list(xs),
    "cons": lambda x,y: [x] + y,
    "car": lambda x: x[0],
    "cdr": lambda x: x[1:],
    "null?": lambda x: x == [],
    "pair?": _is_pair,
    "list?": _is_list,
    "not": lambda x: x is False,
}

In [ ]:
global_env = Env(globals(), parent=None, primitives=_builtin)

We want our tiny lisp implemention to interop with symbols in python so the top level "frame" is globals()

### Special Forms

Special forms look like procedure calls but do not evaluate all arguments eagerly —
each form decides when and how to evaluate its sub-expressions.

| Form | Syntax | Meaning |
|---|---|---|
| `quote` | `'x` | Return expression unevaluated |
| `if` | `(if test then else)` | Conditional; only the taken branch is evaluated |
| `cond` | `(cond (test expr) … (else expr))` | Multi-branch conditional |
| `and` | `(and e …)` | Short-circuit; returns last value or `#f` |
| `or` | `(or e …)` | Short-circuit; returns first truthy value or `#f` |
| `define` | `(define name val)` / `(define (f a…) body)` | Bind a symbol in the current environment |
| `lambda` | `(lambda (a…) body)` | Anonymous procedure |
| `begin` | `(begin e …)` | Evaluate sequence, return last |
| `set!` | `(set! name val)` | Mutate an existing binding |
| `let` | `(let ((x v) …) body)` | Local bindings (all evaluated in outer env) |
| `let*` | `(let* ((x v) …) body)` | Sequential bindings; each sees the previous |
| `apply` | `(apply fn arg… lst)` | Call `fn` spreading `lst` as its arguments |
| `map` | `(map fn lst…)` | Apply `fn` across one or more lists |
| `filter` | `(filter fn lst)` | Keep elements where `fn` returns truthy |
| `for-each` | `(for-each fn lst…)` | Like `map` but for side effects |
| `macro` | `(macro (a…) body)` | Define a code transformer |
| `quasiquote` | `` `(… ,x ,@xs) `` | Template with `,` splice and `,@` list splice |

In [ ]:
#| export
def _sf_quote(xs, env, sfs): return xs[0]

In [ ]:
#| export
def _sf_begin_tco(xs, env, sfs):
    for o in xs[:-1]: scm_eval_tco(o, env, sfs)
    return Thunk(xs[-1], env)

In [ ]:
#| export
def _body_expr(body): return body[0] if len(body) == 1 else [Symbol("begin")] + body

def _arity(ps, vs):
    if len(vs) == len(ps): return vs
    raise ValueError(f"incorrect number of vs passed (wanted: {len(ps)}, got: {len(vs)})")

def _bind_params(ps, vs):
    if isinstance(ps, Symbol): return {ps.s: list(vs)}
    return dict(zip([p.s for p in ps], _arity(ps, vs)))

In [ ]:
#| export
def _sf_if_tco(xs, env, sfs):
    cond,then_,else_ = xs
    br = else_ if scm_eval_tco(cond, env, sfs) is False else then_
    return Thunk(br, env)

In [ ]:
#| export
def _sf_lambda_tco(xs, env, sfs):
    params, *body = xs
    def procedure(*args):
        return Thunk(_body_expr(body), env.new_frame(_bind_params(params, args)))
    return procedure

In [ ]:
#| export
def _sf_define_tco(xs, env, sfs):
    def _mkfn(sym, expr): 
        if not isinstance(sym, Symbol): raise SyntaxError(f"{sym} must be a symbol")
        if env.is_primitive(sym.s): raise SyntaxError(f"cannot redefine primitive '{sym.s}'")
        env[sym.s] = scm_eval_tco(expr, env, sfs)
        return sym.s

    arg0, *rest = xs
    if isinstance(arg0, Symbol): return _mkfn(arg0, _body_expr(rest))
    if isinstance(arg0, list): return _mkfn(arg0[0], [Symbol("lambda"), arg0[1:]] + rest)

In [ ]:
#| export
def _sf_set(xs, env, sfs):
    sym, expr = xs
    if not isinstance(sym, Symbol): raise SyntaxError(f"set! argument {sym} must be a symbol")
    e = env.find(sym.s)
    if e is None: raise NameError(sym.s)

    e[sym.s] = scm_eval_tco(expr, env, sfs)
    return sym.s

In [ ]:
#| export
def _sf_let(xs, env, sfs):
    binds, *body = xs
    new_env = env.new_frame({name.s: scm_eval_tco(val, env, sfs) for name, val in binds})
    return Thunk(_body_expr(body), new_env)

In [ ]:
#| export
def _sf_cond(xs, env, sfs):
    for test, *body in xs:
        if (isinstance(test, Symbol) and test.s == 'else') or scm_eval_tco(test, env, sfs) is not False:
            return Thunk(_body_expr(body), env)

In [ ]:
#| export
def _sf_and(xs, env, sfs):
    if not xs: return True
    for o in xs[:-1]:
        if scm_eval_tco(o, env, sfs) is False: return False
    return Thunk(xs[-1], env)

def _sf_or(xs, env, sfs):
    if not xs: return False
    for o in xs[:-1]:
        v = scm_eval_tco(o, env, sfs)
        if v is not False: return v
    return Thunk(xs[-1], env)

In [ ]:
#| export
def _sf_let_star(xs, env, sfs):
    binds, *body = xs
    for name, val in binds:
        env = env.new_frame({name.s: scm_eval_tco(val, env, sfs)})
    return Thunk(_body_expr(body), env)

In [ ]:
#| export
def _invoke(fn, args, sfs):
    "call fn resolving any TCO thunk"
    r = scm_apply(fn, args)
    return scm_eval_tco(r.expr, r.env, sfs) if isinstance(r, Thunk) else r

def _sf_apply(xs, env, sfs):
    fn   = scm_eval_tco(xs[0], env, sfs)
    args = [scm_eval_tco(o, env, sfs) for o in xs[1:-1]]
    last = scm_eval_tco(xs[-1], env, sfs)
    return _invoke(fn, args + list(last), sfs)

def _sf_map(xs, env, sfs):
    fn   = scm_eval_tco(xs[0], env, sfs)
    lsts = [scm_eval_tco(l, env, sfs) for l in xs[1:]]
    return [_invoke(fn, list(args), sfs) for args in zip(*lsts)]

def _sf_filter(xs, env, sfs):
    fn  = scm_eval_tco(xs[0], env, sfs)
    lst = scm_eval_tco(xs[1], env, sfs)
    return [x for x in lst if _invoke(fn, [x], sfs) is not False]

def _sf_for_each(xs, env, sfs):
    fn   = scm_eval_tco(xs[0], env, sfs)
    lsts = [scm_eval_tco(l, env, sfs) for l in xs[1:]]
    for args in zip(*lsts): _invoke(fn, list(args), sfs)

In [ ]:
#| export
def _macro(args, env, sfs):
    params, *body = args
    return Macro(lambda *vals: scm_eval_tco(_body_expr(body), env.new_frame(_bind_params(params, vals)), sfs))

In [ ]:
#| export
def _qq(x, env, sfs, depth=1):
    "handle ` , and ,@"
    QQ, UQ, UQS = Symbol("quasiquote"), Symbol("unquote"), Symbol("unquote-splicing")
    def tagged(o, sym): return isinstance(o, list) and len(o) == 2 and o[0] == sym

    if not isinstance(x, list): return x
    if tagged(x, QQ): return [QQ, _qq(x[1], env, sfs, depth+1)]
    if tagged(x, UQ):
        return scm_eval_tco(x[1], env, sfs) if depth == 1 else [UQ, _qq(x[1], env, sfs, depth-1)]
    if tagged(x, UQS):
        if depth == 1: raise SyntaxError("unquote-splicing only valid inside a list")
        return [UQS, _qq(x[1], env, sfs, depth-1)]

    def walk(lst):
        result = []
        for o in lst:
            if tagged(o, UQS) and depth == 1: result.extend(scm_eval_tco(o[1], env, sfs))
            else: result.append(_qq(o, env, sfs, depth))
        return result
    return walk(x)

In [ ]:
#| export
def _sf_quasiquote(args, env, sfs): return _qq(args[0], env, sfs)

### Python interop helper

In [ ]:
#| export

class LispCtx:
    "convenenience lisp context for interop"
    def __init__(self):
        self.env = Env(globals(), primitives=_builtin)
    
    def __rmatmul__(self, s):
        sfs = {
            "begin":    _sf_begin_tco,
            "if":       _sf_if_tco,
            "cond":     _sf_cond,
            "and":      _sf_and,
            "or":       _sf_or,
            "define":   _sf_define_tco,
            "lambda":   _sf_lambda_tco,
            "macro":    _macro,
            "set!":     _sf_set,
            "let":      _sf_let,
            "let*":     _sf_let_star,
            "apply":    _sf_apply,
            "map":      _sf_map,
            "filter":   _sf_filter,
            "for-each": _sf_for_each,
            "quote":      _sf_quote,
            "quasiquote": _sf_quasiquote,
        }
        return scm_eval_tco(parse(s), self.env, sfs)
        
lisp = LispCtx()

In [ ]:
from IPython.core.magic import register_cell_magic

register_cell_magic('lisp')(lambda line, cell: cell @ lisp)

<function __main__.<lambda>(line, cell)>

### Examples

**quote** — return unevaluated

In [ ]:
%%lisp
'(1 2 3)

[1, 2, 3]

**define** — bind a symbol; **lambda** — anonymous procedure

In [ ]:
%%lisp
(begin
  (define x 42)
  x)

42

In [ ]:
%%lisp
(begin
  (define (square x) (* x x))
  (square 5))

25

**begin** — evaluate a sequence, return the last

In [ ]:
%%lisp
(begin 1 2 3)

3

**if** — conditional; only the taken branch is evaluated

In [ ]:
%%lisp
(if #t 42 0)

42

**cond** — multi-branch conditional

In [ ]:
%%lisp
(cond
  ((= 1 2) "no")
  ((= 1 1) "yes")
  (else    "other"))

'yes'

**and / or** — short-circuit; return last evaluated value

In [ ]:
%%lisp
(and 1 2 3)

3

In [ ]:
%%lisp
(or #f #f 42)

42

In [ ]:
%%lisp
(and #f (/ 1 0))  ; short-circuits — (/ 1 0) never evaluated

False

In [ ]:
%%lisp
(or 42 (/ 1 0))   ; short-circuits — (/ 1 0) never evaluated

42

**set!** — mutate an existing binding

In [ ]:
%%lisp
(begin
  (define x 1)
  (set! x 42)
  x)

42

**let** — local bindings

In [ ]:
%%lisp
(let ((x 40))
  (+ x 2))

42

**let*** — sequential bindings; each binding sees the previous

In [ ]:
%%lisp
(let* ((x 2) (y (* x 3)))
  (+ x y))

8

**tail calls** — deep recursion without stack overflow

In [ ]:
%%lisp
(begin
  (define (fact n acc)
    (if (= n 0) acc (fact (- n 1) (* n acc))))
  (fact 1000 1))

4023872600770937735437024339230039857193748642107146325437999104299385123986290205920442084869694048004799886101971960586316668729948085589013238296699445909974245040870737599188236277271887325197795059509952761208749754624970436014182780946464962910563938874378864873371191810458257836478499770124766328898359557354325131853239584630755574091142624174743493475534286465766116677973966688202912073791438537195882498081268678383745597317461360853795345242215865932019280908782973084313928444032812315586110369768013573042161687476096758713483120254785893207671691324484262361314125087802080002616831510273418279777047846358681701643650241536913982812648102130927612448963599287051149649754199093422215668325720808213331861168115536158365469840467089756029009505376164758477284218896796462449451607653534081989013854424879849599533191017233555566021394503997362807501378376153071277619268490343526252000158885351473316117021039681759215109077880193931781141945452572238655414610628921879602238389714760

**apply** — call a function with a list as its arguments

In [ ]:
%%lisp
(apply + '(1 2 3 4))

10

**map / filter / for-each** — list processing

In [ ]:
%%lisp
(map (lambda (x) (* x x)) '(1 2 3 4 5))

[1, 4, 9, 16, 25]

In [ ]:
%%lisp
(filter (lambda (x) (> x 2)) '(1 2 3 4 5))

[3, 4, 5]

**lists** — car, cdr, cons

In [ ]:
%%lisp
(car '(10 20 30))

10

In [ ]:
%%lisp
(cdr '(10 20 30))

[20, 30]

In [ ]:
%%lisp
(cons 1 '(2 3))

[1, 2, 3]

**macros** — transform code before evaluation

In [ ]:
%%lisp
(begin
  (define unless (macro (test body) `(if ,test #f ,body)))
  (unless (= 1 2) 42))

42

**quasiquote** — template with selective splicing

In [ ]:
%%lisp
`(a ,(+ 1 2) ,@'(4 5))

[compact.types.Symbol(s='a'), 3, 4, 5]

**python interop** — lisp expressions can reference python symbols

In [ ]:
a = 5
"""
(+ a 6)
""" @ lisp

11

### nbdev postscript

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()